In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyodbc
import sys
import os
import time
from datetime import date
import re
import warnings as wr
wr.filterwarnings('ignore')

In [8]:
seasons_as10 = {1:'Kharif', 2:'Phase-I', 3:'Phase-II', 4:'Phase-III', 5:'Summer'}
seasons_as20 = {1:'Kharif', 2:'E.Kharif', 3:'L.Kharif', 4:'Rabi', 5:'Summer'}

In [171]:
conn_str = (
    r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
    f'DBQ={"./software/RC123kharif_TN.accdb"};'
)
conn = pyodbc.connect(conn_str)

#get data from the table - RC1
state_cd = 33
year = 2024
planning = 260
b4_cod = 250
season_cd = 2
ddg = 2510
ddtrs = 2810
cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
       'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
       'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
       'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']

sql_query1 = 'SELECT * FROM "Copy Of RC1"'
df1 = pd.read_sql(sql_query1, conn)[cols1]
conn.close()

df1 = (df1
    .drop_duplicates()
    .apply(pd.to_numeric, errors='ignore')
    .assign(
        REJ = lambda x : np.where(x['REJ']==1, 1, 0)
        ,STAT = state_cd
        ,SESON = season_cd
        ,NOVPC1 = lambda x : x['NOVPC'].fillna(1)
        ,NOVPC = lambda x : np.where(x['NOVPC1']==0, 1, x['NOVPC1'])
        ,NOVTRS1 = lambda x : x['NOVTRS'].fillna(1)
        ,NOVTRS = lambda x : np.where(x['NOVTRS1']==0, 1, x['NOVTRS1'])
        ,CADC = lambda x : np.where(np.isin(x['CADC'], [1,2,3,4]), x['CADC'], 9)
        ,TLC = lambda x : np.where(x['CADC']==3, 0, np.where(np.isin(x['CADC'], [4,9]), 8, x['TLC']))
        ,MAC = lambda x : np.where(x['MAC']==1, 1, 0)
        ,MUC = lambda x : np.where(x['MAC']==1, x['MUC'], None)
        ,TRSSC = lambda x : np.where(x['TRSSC']==3, 8, x['TRSSC'])
        ,NOVP = planning
        ,DDG = ddg
        ,DDTRS = ddtrs
        ,TCROPA = lambda x : x['TCROPA'].fillna(0)
        ,GEOA = lambda x : x['GEOA'].fillna(0)
        ,YR = year+1
        ,NOVCOD = b4_cod
        ,LFOG = lambda x : np.where(~np.isin(x['LFOG'], [0,1]), None, x['LFOG'])
        ,ROG = lambda x : np.where(~np.isin(x['ROG'], [0,1]), None, x['ROG'])
        ,TRSSF = lambda x : np.where(x['TRSSF'].isna(), None, x['TRSSF'])
    )
    .drop(columns = ['NOVPC1','NOVTRS1'])
)

non_rej_rc1 = df1[rc1['REJ']!=1]
rej_rc1 = df1[rc1['REJ']==1].assign(
	EPC = None
	,NOVPC = None
	,NOVTRS = None
	,CADC = None
	,TLC = None
	,MAC = None
	,MUC = None
	,DDG = None
	,SGC = None
	,GCC = None
	,LFOG = None
	,ROG = None
	,DDTRS = None
	,TRSSC = None
	,TRSSF = None
	,DCKC = None
	,HSSN = None
	,GEOA = None
	,TCROPA = None
	,TNSSN = None
	,TGEOASN = None
	,RC61 = None
	,RC62 = None
	,RC63 = None
)
rc1 = pd.concat([non_rej_rc1, rej_rc1], ignore_index=True)

if rc1['HSSN'].isna().sum() > 0:
    print('='*50)
    print('In RC1, HSSN is Blank. Please Correct...')
    print('='*50)
if rc1[rc1['HSSN'] == 0].shape[0] > 0:
    print('='*50)
    print('In RC1, HSSN is Zero. Please Correct...')
    print('='*50)
if rc1['GEOA'].isna().sum() > 0:
    print('='*50)
    print('In RC1, GEOA is Blank. Please Correct...')
    print('='*50)
if rc1[rc1['GEOA'] == 0].shape[0] > 0:
    print('='*50)
    print('In RC1, GEOA is Zero. Please Correct...')
    print('='*50)


rc1.to_excel('RC1237.xlsx', 'RC1', index=False)

In RC1, HSSN is Zero. Please Correct...
In RC1, GEOA is Zero. Please Correct...


In [ ]:
def clean_rc123(year_int, season_cd, planning, b4_cod):
    #set season name
    season_nm = seasons[season_cd]
    st_cd = 1
    state_cd = 33
    year = 2024
    planning = 260
    b4_cod = 250
    season_cd = 2
    ddg = 2510
    ddtrs = 2810

    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if season_nm.lower() in file_path.name.lower():
                accdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={"./software/RC123kharif_TN.accdb"};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - RC1
    cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
           'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
           'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
           'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']
    
    sql_query1 = 'SELECT * FROM "Copy Of RC1"'
    df1 = pd.read_sql(sql_query1, conn)[cols1]
    
    df1 = (df1
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .assign(
            REJ = lambda x : np.where(x['REJ']==1, 1, 0)
            ,STAT = state_cd
            ,SESON = season_cd
            ,NOVPC1 = lambda x : x['NOVPC'].fillna(1)
            ,NOVPC = lambda x : np.where(x['NOVPC1']==0, 1, x['NOVPC1'])
            ,NOVTRS1 = lambda x : x['NOVTRS'].fillna(1)
            ,NOVTRS = lambda x : np.where(x['NOVTRS1']==0, 1, x['NOVTRS1'])
            ,CADC = lambda x : np.where(np.isin(x['CADC'], [1,2,3,4]), x['CADC'], 9)
            ,TLC = lambda x : np.where(x['CADC']==3, 0, np.where(np.isin(x['CADC'], [4,9]), 8, x['TLC']))
            ,MAC = lambda x : np.where(x['MAC']==1, 1, 0)
            ,MUC = lambda x : np.where(x['MAC']==1, x['MUC'], None)
            ,TRSSC = lambda x : np.where(x['TRSSC']==3, 8, x['TRSSC'])
            ,NOVP = planning
            ,DDG = ddg
            ,DDTRS = ddtrs
            ,TCROPA = lambda x : x['TCROPA'].fillna(0)
            ,GEOA = lambda x : x['GEOA'].fillna(0)
            ,YR = year+1
            ,NOVCOD = b4_cod
            ,LFOG = lambda x : np.where(~np.isin(x['LFOG'], [0,1]), None, x['LFOG'])
            ,ROG = lambda x : np.where(~np.isin(x['ROG'], [0,1]), None, x['ROG'])
            ,TRSSF = lambda x : np.where(x['TRSSF'].isna(), None, x['TRSSF'])
        )
        .drop(columns = ['NOVPC1','NOVTRS1'])
    )
    
    non_rej_rc1 = df1[rc1['REJ']!=1]
    rej_rc1 = df1[rc1['REJ']==1].assign(
    	EPC = None
    	,NOVPC = None
    	,NOVTRS = None
    	,CADC = None
    	,TLC = None
    	,MAC = None
    	,MUC = None
    	,DDG = None
    	,SGC = None
    	,GCC = None
    	,LFOG = None
    	,ROG = None
    	,DDTRS = None
    	,TRSSC = None
    	,TRSSF = None
    	,DCKC = None
    	,HSSN = None
    	,GEOA = None
    	,TCROPA = None
    	,TNSSN = None
    	,TGEOASN = None
    	,RC61 = None
    	,RC62 = None
    	,RC63 = None
    )
    
    rc1 = pd.concat([non_rej_rc1, rej_rc1], ignore_index=True)

    if rc1['HSSN'].isna().sum() > 0:
        print('='*50)
        print('In RC1, HSSN is Blank. Please Correct...')
        print('='*50)
    if rc1[rc1['HSSN'] == 0].shape[0] > 0:
        print('='*50)
        print('In RC1, HSSN is Zero. Please Correct...')
        print('='*50)
    if rc1['GEOA'].isna().sum() > 0:
        print('='*50)
        print('In RC1, GEOA is Blank. Please Correct...')
        print('='*50)
    if rc1[rc1['GEOA'] == 0].shape[0] > 0:
        print('='*50)
        print('In RC1, GEOA is Zero. Please Correct...')
        print('='*50)
        
    rc1.to_excel('RC1237_clean.xlsx', 'RC1', index=False)






    
    #get data from the table - RC2
    cols2 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA', 'VILL', 'EPC', 
             'HSSN', 'TNSSN', 'CROP', 'VARC', 'ARSU', 'ARSI', 'ARPU', 'ARPI']
    
    sql_query2 = 'SELECT * FROM "Copy Of RC2"'
    df2 = pd.read_sql(sql_query2, conn)[cols2]
    df2 = (df2
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .assign(
            RC = 2
            ,ST = st_cd
            ,YR = year+1
            ,SESON = season_cd
            ,STAT = state_cd
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    rc2 = df2.copy()
    rc2.to_excel('RC1237_clean.xlsx', 'RC2', index=False)







    
    #get data from the table - RC3
    cols3 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA',
             'VILL', 'SN', 'CROP', 'VARC', 'IRRC', 'ERC']
    
    sql_query3 = 'SELECT * FROM "Copy Of RC3"'
    df3 = pd.read_sql(sql_query3, conn)[cols3]
    df3 = (df3
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .assign(
            RC = 3
            ,ST = st_cd
            ,YR = year+1
            ,SESON = season_cd
            ,STAT = state_cd
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    rc3 = df3.copy()
    rc3.to_excel('RC1237_clean.xlsx', 'RC3', index=False)
    
    
    conn.close()
    return rc1, rc2, rc3

In [ ]:
def clean_rc7(year_int):
    yr_int = year_int - 2001
    year = str(year_int)
    
    mdb_path = r"C:/ICS_Scheme/AS2.0/AS"+year+"/NSSAS"+year+".mdb"
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={mdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - Sch20RC7
    sql_query = "SELECT * FROM Sch20RC7"
    rc7 = (
        pd
        .read_sql(sql_query, conn)
        .astype({'EXPTID' : 'int64'})
    )
    conn.close()
    return rc7

In [3]:
def granularize(rc1, rc2, rc3, rc4):
    pass

In [4]:
def generate_annex(grc1, grc2, grc3, grc4):
    pass

In [ ]:
if __name__ == "__main__":
    #read & clean RCs
    year_int = 2024            #2024 for A.Y. 2024-25
    season_cd = [2,3]              #1-Kharif, 4-Rabi
    rc1_novp = {
        1 : [0,0],                     ##season_1 planning [cen, state]
        2 : [260, 260],                ##season_2 planning [cen, state]
        3 : [260, 260],                ##season_3 planning [cen, state]
        4 : [260, 260],                ##season_4 planning [cen, state]
        5 : [0,0]                      ##season_5 planning [cen, state]
    }
    rc1_novcod = {
        1 : [0,0],                     ##season_1 NOVCOD [cen, state]
        2 : [260, 260],                ##season_2 NOVCOD [cen, state]
        3 : [260, 260],                ##season_3 NOVCOD [cen, state]
        4 : [260, 260],                ##season_4 NOVCOD [cen, state]
        5 : [0,0]                      ##season_5 NOVCOD [cen, state]
    }

    rc1, rc2, rc3 = clean_rc123(year_int, season_cd, rc1_novp, rc1_novcod)
    rc7 = clean_rc7(year_int, season_cd)

    #Granularize RCs
    grc1, grc2, grc3, grc7 = granularize(crc1, crc2, crc3, crc7)

    #Generate Annexure Data
    generate_annex(grc1, grc2, grc3, grc7)